<a href="https://colab.research.google.com/github/fralfaro/MAT281/blob/main/docs/labs/lab_03.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# MAT281 - Laboratorio N°03





**Objetivo**: Aplicar técnicas avanzadas de manipulación y análisis de datos con pandas sobre un conjunto real de datos de contenido de Netflix, reforzando buenas prácticas y métodos eficientes sin recurrir a `groupby`, `merge`, `pivot`, ni `join`.



**Dataset**:

Trabajaremos con el archivo `netflix_titles.csv`, que contiene información sobre los títulos disponibles en la plataforma Netflix hasta el año 2021.

| Variable       | Clase     | Descripción                                                                 |
|----------------|-----------|------------------------------------------------------------------------------|
| show_id        | caracter  | Identificador único del título en el catálogo de Netflix.                   |
| type           | caracter  | Tipo de contenido: 'Movie' o 'TV Show'.                                     |
| title          | caracter  | Título del contenido.                                                       |
| director       | caracter  | Nombre del director (puede ser nulo).                                       |
| cast           | caracter  | Lista de actores principales (puede ser nulo).                              |
| country        | caracter  | País o países donde se produjo el contenido.                                |
| date_added     | fecha     | Fecha en la que el título fue agregado al catálogo de Netflix.              |
| release_year   | entero    | Año de lanzamiento original del título.                                     |
| rating         | caracter  | Clasificación por edad (por ejemplo: 'PG-13', 'TV-MA').                      |
| duration       | caracter  | Duración del contenido (minutos o número de temporadas para series).        |
| listed_in      | caracter  | Categorías o géneros en los que está clasificado el contenido.              |
| description    | caracter  | Breve sinopsis del contenido.                                               |




In [184]:
import pandas as pd

# Cargar datos
df = pd.read_csv('https://raw.githubusercontent.com/fralfaro/MAT281/main/docs/labs/data/netflix_titles.csv')
df.head()

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...
3,s4,TV Show,Jailbirds New Orleans,NaN,NaN,NaN,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo..."
4,s5,TV Show,Kota Factory,NaN,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...



### Parte 1: Limpieza y preparación

1. Revisar y describir el dataset:

   * ¿Cuántas filas y columnas tiene?
   * ¿Qué tipos de datos hay?
   * ¿Cuántos valores nulos hay por columna?

2. Transformar la columna `date_added` a tipo fecha.

3. Crear columnas auxiliares con `assign`:

   * Año (`year_added`)
   * Mes (`month_added`)



In [185]:
#columnas y filas
print('El dataset contiene', len(df), 'filas y', len(df.columns), 'columnas')

#tipo de datos
df.dtypes


El dataset contiene 8807 filas y 12 columnas


,0
show_id,object
type,object
title,object
director,object
cast,object
country,object
date_added,object
release_year,int64
rating,object
duration,object


In [186]:
#valores nulos
df.isnull().sum()

,0
show_id,0
type,0
title,0
director,2634
cast,825
country,831
date_added,10
release_year,0
rating,4
duration,3


## Parte 2: Técnicas avanzadas de pandas

4. Utilizar `.loc` para seleccionar películas (`type == 'Movie'`) que fueron agregadas después del año 2018.

5. Utilizar `str.contains()` y `str.extract()`:

   * Filtrar títulos que contienen la palabra 'love' (sin distinguir mayúsculas/minúsculas).
   * Extraer la duración en minutos para las películas desde la columna `duration`.

6. Aplicar `explode()` sobre la columna `listed_in` para obtener una fila por cada género.

7. Obtener un top 10 de géneros más frecuentes utilizando `value_counts()`.

8. Aplicar `where()` y `mask()` para marcar las películas de más de 120 minutos como contenido largo en una nueva columna.

9. Utilizar `.loc` para filtrar películas que cumplen con:

   * Más de 100 minutos de duración.
   * Rating igual a `'R'`.
   * País igual a `'United States'`.

10. Utilizar `.style` para formatear visualmente el top 10 de películas más largas.

In [187]:
#4
df_new = df.loc[(df['release_year'] >= 2018) & (df['type'] == 'Movie')]
df_new.head()

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
6,s7,Movie,My Little Pony: A New Generation,"Robert Cullen, José Luis Ucha","Vanessa Hudgens, Kimiko Glenn, James Marsden, ...",NaN,"September 24, 2021",2021,PG,91 min,Children & Family Movies,Equestria's divided. But a bright-eyed hero be...
9,s10,Movie,The Starling,Theodore Melfi,"Melissa McCarthy, Chris O'Dowd, Kevin Kline, T...",United States,"September 24, 2021",2021,PG-13,104 min,"Comedies, Dramas",A woman adjusting to life after a loss contend...
12,s13,Movie,Je Suis Karl,Christian Schwochow,"Luna Wedler, Jannis Niewöhner, Milan Peschel, ...","Germany, Czech Republic","September 23, 2021",2021,TV-MA,127 min,"Dramas, International Movies",After most of her family is murdered in a terr...
13,s14,Movie,Confessions of an Invisible Girl,Bruno Garotti,"Klara Castanho, Lucca Picon, Júlia Gomes, Marc...",NaN,"September 22, 2021",2021,TV-PG,91 min,"Children & Family Movies, Comedies",When the clever but socially-awkward Tetê join...


In [188]:
#5
df_new = df.loc[df['title'].str.contains(' Love ', case=False)]
df_new.head()


,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
506,s507,Movie,This Little Love Of Mine,Christine Luby,"Saskia Hampele, Liam McIntyre, Lynn Gilmartin,...",Australia,"July 7, 2021",2021,TV-G,92 min,"International Movies, Romantic Movies",A workaholic lawyer returns to her island home...
659,s660,TV Show,Bangkok Love Stories: Innocence,NaN,"Nida Patcharaveerapong, Nicole Theriault, Natt...",Thailand,"June 19, 2021",2018,TV-14,1 Season,"International TV Shows, Romantic TV Shows, TV ...",From a teenage parkour enthusiast to a bawdy r...
849,s850,Movie,Sam Smith: Love Goes - Live at Abbey Road Studios,NaN,Sam Smith,NaN,"May 22, 2021",2020,TV-G,61 min,"International Movies, Music & Musicals",Grammy-winning artist Sam Smith gives an intim...
1161,s1162,Movie,Elizabeth and Margaret: Love and Loyalty,NaN,NaN,United Kingdom,"March 26, 2021",2020,TV-PG,87 min,Documentaries,This documentary takes an intimate look at the...
1428,s1429,Movie,Is Love Enough? Sir,Rohena Gera,"Tillotama Shome, Vivek Gomber, Geetanjali Kulk...","India, France","January 8, 2021",2018,TV-MA,99 min,"Dramas, Independent Movies, International Movies",A young widow is hired as the domestic helper ...


In [189]:
temp = df['duration'].str.contains('min', case=False, na=False)

df_new = df.loc[temp].copy()

df_new["duration_min"] = df_new["duration"].str.extract(r"(\d+)").astype(float)

df_new.head()
print(df_new["duration_min"])

0        90.0
6        91.0
7       125.0
9       104.0
12      127.0
        ...  
8801     96.0
8802    158.0
8804     88.0
8805     88.0
8806    111.0
Name: duration_min, Length: 6128, dtype: float64


In [190]:
#6
df['listed_in'] = df['listed_in'].str.split(', ')

df_exploded = df.explode('listed_in')

df_exploded.head()


,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,International TV Shows,"After crossing paths at a party, a Cape Town t..."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,TV Dramas,"After crossing paths at a party, a Cape Town t..."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,TV Mysteries,"After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,"September 24, 2021",2021,TV-MA,1 Season,Crime TV Shows,To protect his family from a powerful drug lor...


In [191]:
#7
df_new7 = df_exploded['listed_in'].value_counts()#.sort_index()
print(df_new7.head(10))

listed_in
International Movies        2752
Dramas                      2427
Comedies                    1674
International TV Shows      1351
Documentaries                869
Action & Adventure           859
TV Dramas                    763
Independent Movies           756
Children & Family Movies     641
Romantic Movies              616
Name: count, dtype: int64


In [192]:
#8
df["duration_min"] = df["duration"].str.extract(r"(\d+)").astype(float)
df["tipe_duration"] = "Corto"
df["tipe_duration"] = df["tipe_duration"].mask(df["duration_min"] > 120, "Largo")
print(df["tipe_duration"])

0       Corto
1       Corto
2       Corto
3       Corto
4       Corto
        ...  
8802    Largo
8803    Corto
8804    Corto
8805    Corto
8806    Corto
Name: tipe_duration, Length: 8807, dtype: object


In [193]:
#9
df_filtro = df.loc[
    (df["duration_min"] > 100) &
    (df["rating"] == "R") &
    (df["country"] == "United States")
]
df_filtro.head()


,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description,duration_min,tipe_duration
48,s49,Movie,Training Day,Antoine Fuqua,"Denzel Washington, Ethan Hawke, Scott Glenn, T...",United States,"September 16, 2021",2001,R,122 min,"[Dramas, Thrillers]",A rookie cop with one day to prove himself to ...,122.0,Largo
81,s82,Movie,Kate,Cedric Nicolas-Troyan,"Mary Elizabeth Winstead, Jun Kunimura, Woody H...",United States,"September 10, 2021",2021,R,106 min,[Action & Adventure],"Slipped a fatal poison on her final job, a rut...",106.0,Corto
131,s132,Movie,Blade Runner: The Final Cut,Ridley Scott,"Harrison Ford, Rutger Hauer, Sean Young, Edwar...",United States,"September 1, 2021",1982,R,117 min,"[Action & Adventure, Classic Movies, Cult Movies]","In a smog-choked dystopian Los Angeles, blade ...",117.0,Corto
139,s140,Movie,Do the Right Thing,Spike Lee,"Danny Aiello, Ossie Davis, Ruby Dee, Richard E...",United States,"September 1, 2021",1989,R,120 min,"[Classic Movies, Comedies, Dramas]","On a sweltering day in Brooklyn, simmering rac...",120.0,Corto
144,s145,Movie,House Party,Reginald Hudlin,"Christopher Reid, Christopher Martin, Robin Ha...",United States,"September 1, 2021",1990,R,104 min,"[Comedies, Cult Movies]","Grounded by his strict father, Kid risks life ...",104.0,Corto


In [194]:
#10
top10 = df.sort_values(by="duration_min", ascending=False).head(10)

top10.style.background_gradient(subset=["duration_min"], cmap="Greens")


,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description,duration_min,tipe_duration
4253,s4254,Movie,Black Mirror: Bandersnatch,nan,"Fionn Whitehead, Will Poulter, Craig Parkinson, Alice Lowe, Asim Chaudhry",United States,"December 28, 2018",2018,TV-MA,312 min,"['Dramas', 'International Movies', 'Sci-Fi & Fantasy']","In 1984, a young programmer begins to question reality as he adapts a dark fantasy novel into a video game. A mind-bending tale with multiple endings.",312.000000,Largo
717,s718,Movie,Headspace: Unwind Your Mind,nan,"Andy Puddicombe, Evelyn Lewis Prieto, Ginger Daniels, Darren Pettie, Simon Prebble, Rhiannon Mcgavin, Kate Seftel",nan,"June 15, 2021",2021,TV-G,273 min,['Documentaries'],"Do you want to relax, meditate or sleep deeply? Personalize the experience according to your mood or mindset with this Headspace interactive special.",273.000000,Largo
2491,s2492,Movie,The School of Mischief,Houssam El-Din Mustafa,"Suhair El-Babili, Adel Emam, Saeed Saleh, Younes Shalabi, Hadi El-Gayyar, Ahmad Zaki, Hassan Moustafa",Egypt,"May 21, 2020",1973,TV-14,253 min,"['Comedies', 'Dramas', 'International Movies']",A high school teacher volunteers to transform five notorious misfits into model students — and has unintended results.,253.000000,Largo
2487,s2488,Movie,No Longer kids,Samir Al Asfory,"Said Saleh, Hassan Moustafa, Ahmed Zaki, Younes Shalabi, Nadia Shukri, Karima Mokhtar",Egypt,"May 21, 2020",1979,TV-14,237 min,"['Comedies', 'Dramas', 'International Movies']","Hoping to prevent their father from skipping town with his mistress, four rowdy siblings resort to absurd measures to stop him.",237.000000,Largo
2484,s2485,Movie,Lock Your Girls In,Fouad El-Mohandes,"Fouad El-Mohandes, Sanaa Younes, Sherihan, Ahmed Rateb, Ijlal Zaki, Zakariya Mowafi",nan,"May 21, 2020",1982,TV-PG,233 min,"['Comedies', 'International Movies', 'Romantic Movies']",A widower believes he must marry off his three problematic daughters before he can pursue his real goal of marrying his secret love.,233.000000,Largo
2488,s2489,Movie,Raya and Sakina,Hussein Kamal,"Suhair El-Babili, Shadia, Abdel Moneim Madbouly, Ahmed Bedir",nan,"May 21, 2020",1984,TV-14,230 min,"['Comedies', 'Dramas', 'International Movies']","When robberies and murders targeting women sweep early 20th-century Egypt, the hunt for suspects leads to two shadowy sisters. Based on a true story.",230.000000,Largo
166,s167,Movie,Once Upon a Time in America,Sergio Leone,"Robert De Niro, James Woods, Elizabeth McGovern, Treat Williams, Tuesday Weld, Burt Young, Joe Pesci, Danny Aiello, William Forsythe, James Hayden","Italy, United States","September 1, 2021",1984,R,229 min,"['Classic Movies', 'Dramas']",Director Sergio Leone's sprawling crime epic follows a group of Jewish mobsters who rise in the ranks of organized crime in 1920s New York City.,229.000000,Largo
7932,s7933,Movie,Sangam,Raj Kapoor,"Raj Kapoor, Vyjayanthimala, Rajendra Kumar, Lalita Pawar, Achala Sachdev, Hari Shivdasani, Raj Mehra, Iftekhar",India,"December 31, 2019",1964,TV-14,228 min,"['Classic Movies', 'Dramas', 'International Movies']","Returning home from war after being assumed dead, a pilot weds the woman he has long loved, unaware that she had been planning to marry his best friend.",228.000000,Largo
1019,s1020,Movie,Lagaan,Ashutosh Gowariker,"Aamir Khan, Gracy Singh, Rachel Shelley, Paul Blackthorne, Kulbhushan Kharbanda, Raghuvir Yadav, Yashpal Sharma, Rajendranath Zutshi, Rajesh Vivek, Aditya Lakhia","India, United Kingdom","April 17, 2021",2001,PG,224 min,"['Dramas', 'International Movies', 'Music & Musicals']","In 1890s India, an arrogant British commander challenges the harshly taxed residents of Champaner to a high-stakes cricket match.",224.000000,Largo
4573,s4574,Movie,Jodhaa Akbar,Ashutosh Gowariker,"Hrithik Roshan, Aishwarya Rai Bachchan, Sonu Sood, Poonam Sinha, Suhasini Mulay, Ila Arun, Raza Murad, Kulbhushan Kharbanda, Abeer Abrar",India,"October 1, 2018",200



### Pregunta Desafío

11. ¿Cuáles son las combinaciones más frecuentes de género y rating en el dataset?
    (Sugerencia: utilizar `value_counts` con `subset=["genre", "rating"]` después de aplicar `explode()`).



### Bonus: Análisis de duplicados y limpieza

12. ¿Existen películas con el mismo nombre (`title`) pero con distinto año de lanzamiento (`release_year`)?
13. ¿Cuántos títulos únicos hay en total en la columna `title`?





In [203]:
#11

combs = df_exploded.value_counts(subset=["listed_in", "rating"])

print(combs)

#12
rep_title = df["title"][df["title"].duplicated(keep=False)].unique()
print(rep_title)
#la lista está vacía, por lo que no hay películas con el mismo nombre

#13
titles = df['title'].nunique()
print(titles)
#Todos los títulos son únicos, es decir, hay 8807 en total


listed_in               rating  
International Movies    TV-MA       1130
                        TV-14       1065
Dramas                  TV-MA        830
International TV Shows  TV-MA        714
Dramas                  TV-14        693
                                    ... 
TV Sci-Fi & Fantasy     NR             1
TV Mysteries            TV-G           1
TV Sci-Fi & Fantasy     TV-Y7-FV       1
TV Shows                R              1
Action & Adventure      G              1
Name: count, Length: 309, dtype: int64
[]
8807
8807
